# Ensemble of All Methods — CIFAR-10

Loads checkpoints from all 4 trained models (Baseline CNN, HPO, Evolutionary NAS, DARTS) and combines their predictions via **soft voting** and **hard voting**.

If checkpoints are missing locally, the notebook automatically restores them from `My Drive/GGSN-project/checkpoints/`. Run `notebooks/run_all_experiments.ipynb` first to generate them.

The ensemble config lives in `experiments/ensemble_config.yaml`. The logic lives in `scripts/run_ensemble.py`.

In [ ]:
# If you opened this notebook outside the cloned repository, clone it first.
# Change BRANCH if you want to run a different branch.
import os
from pathlib import Path

REPO_URL = "https://github.com/iwadas/GGSN-project.git"
BRANCH = "main"
REPO_DIR = Path("/content/GGSN-project")

if not Path("pyproject.toml").exists():
    os.chdir("/content")
    if not REPO_DIR.exists():
        !git clone -b {BRANCH} {REPO_URL}
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

## Install Dependencies

In [ ]:
%pip install -q uv
!uv pip install --system -q optuna numpy pandas matplotlib pyyaml tqdm

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## Mount Google Drive

Checkpoints are backed up on Drive from `run_all_experiments.ipynb`. This notebook restores them if missing locally.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CKPT = Path("/content/drive/MyDrive/GGSN-project/checkpoints")
print(f"Drive backup: {DRIVE_CKPT}")
print(f"Exists: {DRIVE_CKPT.exists()}")

## Restore Checkpoints from Drive

For each checkpoint — if missing locally, restore from Drive backup.

In [ ]:
import shutil

expected = [
    "checkpoints/baseline_cnn.pt",
    "checkpoints/hpo_best_baseline_cnn.pt",
    "checkpoints/evolutionary_best_cnn.pt",
    "checkpoints/darts_best_cnn.pt",
]

missing = [p for p in expected if not Path(p).exists()]
restored = []
failed = []

for p in missing:
    fname = Path(p).name
    src = DRIVE_CKPT / fname
    if src.exists():
        Path("checkpoints").mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, p)
        restored.append(fname)
        print(f"[RESTORED] {fname} <- Drive")
    else:
        failed.append(fname)
        print(f"[MISSING] {fname} not found locally or on Drive")

if not restored and not failed:
    print("All checkpoints present locally.")

if failed:
    print("\nMissing on Drive too. Run notebooks/run_all_experiments.ipynb first.")
else:
    print("\nReady to run ensemble.")

## Review Config

In [ ]:
!cat experiments/ensemble_config.yaml

## Run Ensemble

In [ ]:
!python scripts/run_ensemble.py --config experiments/ensemble_config.yaml

## Show Results

In [ ]:
import json
from IPython.display import Image, display

summary = json.loads(Path("results/ensemble_summary.json").read_text())
print(json.dumps(summary, indent=2))

In [ ]:
display(Image("plots/ensemble_comparison.png"))

## Backup Results to Drive

In [ ]:
from pathlib import Path

DRIVE_RESULTS = Path("/content/drive/MyDrive/GGSN-project/results")
DRIVE_PLOTS = Path("/content/drive/MyDrive/GGSN-project/plots")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

for f in ["results/ensemble_summary.json", "plots/ensemble_comparison.png"]:
    p = Path(f)
    if p.exists():
        dst = (DRIVE_RESULTS if "results" in f else DRIVE_PLOTS) / p.name
        shutil.copy2(p, dst)
        print(f"[BACKUP] {p.name} -> {dst}")